# ScoutTrainer — GPU pipeline on Colab

Runs the heavy perception pipeline on Colab's free GPU, then hands you a zip of results to view in your **local** dashboard.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Steps:** 1) upload project zip → 2) install → 3) set the video URL → 4) run → 5) download results → unzip into your local project's `data/` folder → open the local dashboard.

In [ ]:
# 1) Upload the project as a zip (zip the scout-agent folder WITHOUT .venv and data/)
from google.colab import files
up = files.upload()  # pick scout-agent.zip
!rm -rf /content/scout && mkdir -p /content/scout
import zipfile, pathlib
zipfile.ZipFile(next(iter(up))).extractall('/content/scout')
# find the project root inside the zip
root = next(p.parent for p in pathlib.Path('/content/scout').rglob('pyproject.toml'))
%cd {root}
!nvidia-smi -L

In [ ]:
# 2) Install (Colab already has torch+CUDA and ffmpeg)
!pip -q install -e ".[perception]" 2>&1 | tail -1
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
# 3) Set your inputs
VIDEO_URL = 'https://www.youtube.com/watch?v=E9GLiV_jfro'  # <-- change me
REF_POINTS = ''   # optional: path to refs.json (upload via left sidebar), else ''
ROSTER = ''       # optional: path to roster.csv, else ''

In [ ]:
# 4) Run the pipeline on GPU
# (the 'project' stage stops if no ref points are given — perception results are still saved)
import os
os.environ['SCOUT_DEVICE'] = 'cuda'
args = ['--url', VIDEO_URL]
if REF_POINTS: args += ['--ref-points', REF_POINTS]
if ROSTER: args += ['--roster', ROSTER]
!python -m scout.pipeline {' '.join(args)}

In [ ]:
# 5) Zip results and download — extract into your LOCAL project's data/ folder,
#    then run `streamlit run app/dashboard.py` locally to browse ratings
!cd data && zip -qr /content/scout_results.zip . -x 'videos/*' 'uploads/*'
from google.colab import files
files.download('/content/scout_results.zip')